In [0]:
from pyspark.sql import functions as F
customer_schema = """
    customer_id STRING,
    email STRING,
    first_name STRING,
    last_name STRING,
    gender STRING,
    street STRING,
    city STRING,
    country_code STRING,
    row_status STRING,
    row_time TIMESTAMP
"""

df_customer = (
    spark.table('dev.multiplex_bronze.kafka_bronze')
        .filter(F.col('topic') == 'customers')
        .select(F.from_json(F.col('value').cast('string'), customer_schema).alias('v'))
        .select('v.*')
        .filter(F.col('row_status').isin(["insert", 'update']))
)

display(df_customer)

In [0]:
# Keep the most recent update / record: use Window function
from pyspark.sql.window import Window
window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())
latest_df = (
    df_customer.withColumn("rank", F.rank().over(window))
        .filter(F.col("rank") == 1)
        .drop('rank')
)

display(latest_df)

In [0]:
ranked_df = (
    spark.readStream.table('dev.multiplex_bronze.kafka_bronze')
        .filter(F.col('topic') == 'customers')
        .select(F.from_json(F.col('value').cast('string'), customer_schema).alias('v'))
        .select('v.*')
        .filter(F.col('row_status').isin(['insert', 'update']))
        .withColumn("rank", F.rank().over(window))
        .filter(F.col('rank') == 1)
        .drop('rank')
)

display(ranked_df, checkpointLocation = "/Volumes/dev/pro_landing_zone/checkpoints/display_kafka_bronze/customers")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
def batch_upsert(microBatchDF, batchId):
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

    (
        microBatchDF.filter(F.col('row_status').isin(["insert", "update"]))
            .withColumn("rank", F.rank().over(window))
            .filter(F.col('rank') == 1)
            .drop("rank")
            .createOrReplaceTempView("ranked_customers")
    )

    sql_query = """
        MERGE INTO dev.silver.customers_silver c
        USING ranked_customers rc
        ON c.customer_id = rc.customer_id
        WHEN MATCHED AND c.row_time < rc.row_time THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """
    microBatchDF.sparkSession.sql(sql_query)


In [0]:
%sql
-- Create the target customers table 
CREATE OR REPLACE TABLE dev.silver.customers_silver (
  customer_id STRING,
  email STRING,
  first_name STRING,
  last_name STRING,
  gender STRING,
  street STRING,
  city STRING,
  country STRING,
  row_time TIMESTAMP
)

In [0]:
# Copy country_lookup json data to raw layer
dbutils.fs.cp("s3://dalhussein-courses/DE-Pro/datasets/bookstore/v1/country_lookup","/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/coutry_lookup/", recurse=True)

In [0]:
df_country_lookup = spark.read.json("/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/coutry_lookup/*")
display(df_country_lookup)

In [0]:
# Now write streaming table customers to silver layer by joininig to the lookup table
df_country_lookup = spark.read.json("/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/coutry_lookup/*")
def process_customers_silver():
    (
        spark.readStream
                .table('dev.multiplex_bronze.kafka_bronze')
                .filter(F.col("topic") == "customers")
                .select(F.from_json(F.col("value").cast("string"), schema=customer_schema).alias('v'))
                .select('v.*')
                .join(
                    F.broadcast(df_country_lookup),
                    F.col("country_code") == F.col("code"),
                    "inner"
                )
            .writeStream
            .foreachBatch(batch_upsert)
            .option("checkpointLocation", "/Volumes/dev/pro_landing_zone/checkpoints/customers_silver")
            .trigger(availableNow=True)
            .start()

    )

In [0]:
row_count = spark.table('dev.silver.customers_silver').count()
unique_count = spark.table('dev.silver.customers_silver').select('customer_id').distinct().count()

assert row_count == unique_count

print("Unit Test has passed :)")
print(f"Total number of records: {row_count} and unique records: {unique_count}")